# Notebook 08 — Optimización de la posición del detector

Pregunta práctica: **¿dónde conviene poner el detector?** Para responderla, recorremos
un grid de posiciones candidatas alrededor del cráter, computamos el muograma DEM en cada
una, y elegimos la posición que maximiza una métrica de "información" — definida como la
varianza de la transmisión $T(\theta, \phi)$ sobre los píxeles del volcán.

La idea: si todas las direcciones son cielo abierto (T≈1) o todas opacas (T≈0), el
muograma no carga información tomográfica. La varianza alta indica contraste — más
señal útil.

**Costo**: ~1 s por posición × 11×11 = 121 posiciones ≈ 2 min wall-clock. Usamos malla
angular más gruesa (1° en vez de 0.5°) y `n_steps=200` para acelerar; la métrica es
robusta frente a esta sub-resolución.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from pyproj import Transformer
from tqdm import tqdm

from analysis.muograma import (
    DEMTerrain, rock_opacity_grid, transmission_map, muograma_information,
)

In [ ]:
# Target volcano + default station (same convention as NB 04 / 05)
TARGET_VOLCAN = "fuego"

with open(REPO_ROOT / "data" / "volcanoes.yaml") as f:
    VOLCAN_CFG = yaml.safe_load(f)[TARGET_VOLCAN]

DEM_FILE = REPO_ROOT / 'data' / 'dem' / 'Copernicus_DSM_COG_10_N14_00_W091_00_DEM.tif'

terrain = DEMTerrain(DEM_FILE,
                     origin_lonlat=(VOLCAN_CFG['lon'], VOLCAN_CFG['lat']),
                     density_gcc=VOLCAN_CFG['cone']['density_gcc'])
summit_z = terrain.summit_elevation

print(f"TARGET_VOLCAN = {TARGET_VOLCAN}")
print(f"  Cráter @ ({VOLCAN_CFG['lat']}, {VOLCAN_CFG['lon']}), cima a {summit_z:.0f} m")
print(f"  Default station (referencia): {VOLCAN_CFG['default_station']}")

## 1. Grid de posiciones candidatas

Cuadrícula de 11×11 = 121 puntos en una ventana de ±10 km alrededor del cráter, en
coordenadas locales UTM (eje X = este, Y = norte, ambos en metros con cero en el cráter).
Cada celda mide 2 km × 2 km.

In [ ]:
# Grid de posiciones candidatas (en coords locales relativas al cráter)
HALF_EXTENT_KM = 10.0
GRID_N = 11

xs_cand = np.linspace(-HALF_EXTENT_KM*1000, HALF_EXTENT_KM*1000, GRID_N)
ys_cand = np.linspace(-HALF_EXTENT_KM*1000, HALF_EXTENT_KM*1000, GRID_N)
XX, YY = np.meshgrid(xs_cand, ys_cand)

print(f"Grid: {GRID_N}×{GRID_N} = {GRID_N*GRID_N} posiciones candidatas")
print(f"Cobertura: ±{HALF_EXTENT_KM} km desde el cráter (paso {2*HALF_EXTENT_KM/(GRID_N-1):.1f} km)")

## 2. Setup de ray tracing (más gruesa para optimización rápida)

`theta_deg ∈ [40°, 85°]` paso 1° → 46 píxeles cenitales.
`phi_deg ∈ [0°, 360°)` paso 5° → 72 píxeles azimutales (rango completo para no asumir
dirección — la malla es del *espacio*, no del campo de visión del detector).
46 × 72 = 3312 píxeles por posición. ~10× más rápido que las mallas finas de NB 05.

In [ ]:
theta_deg_opt = np.arange(40.0, 86.0, 1.0)
phi_deg_opt   = np.arange(0.0, 360.0, 5.0)
theta_rad_opt = np.radians(theta_deg_opt)
phi_rad_opt   = np.radians(phi_deg_opt)
print(f"Malla angular: {len(theta_deg_opt)}×{len(phi_deg_opt)} = {len(theta_deg_opt)*len(phi_deg_opt)} pixeles")

## 3. Loop sobre posiciones — muograma + métrica

Por cada `(det_x, det_y)`: tomar la elevación del terreno + 2 m, calcular L_DEM,
calcular T(θ,φ), evaluar `muograma_information(T, L)` (varianza de T sobre píxeles
del volcán). Se guarda todo en `info_grid` (matriz 11×11).

In [ ]:
info_grid = np.zeros((GRID_N, GRID_N))
det_z_grid = np.zeros((GRID_N, GRID_N))
L_max_grid = np.zeros((GRID_N, GRID_N))

print("Recorriendo grid de posiciones...")
for i in tqdm(range(GRID_N)):
    for j in range(GRID_N):
        det_x_ij = float(XX[i, j])
        det_y_ij = float(YY[i, j])
        # Elevación del terreno en esa posición + 2 m altura del detector
        det_z_ground = float(terrain.elevation_at(np.array(det_x_ij), np.array(det_y_ij)))
        det_z_ij = det_z_ground + 2.0
        det_z_grid[i, j] = det_z_ground

        detector_xyz = (det_x_ij, det_y_ij, det_z_ij)
        L = rock_opacity_grid(terrain, detector_xyz, theta_rad_opt, phi_rad_opt,
                              max_length_m=8_000.0, n_steps=200)
        T = transmission_map(L, theta_rad_opt)
        info_grid[i, j] = muograma_information(T, L, mode='variance_volcano')
        L_max_grid[i, j] = L.max()

print(f"\nValor de información — rango: [{info_grid.min():.4f}, {info_grid.max():.4f}]")
i_best, j_best = np.unravel_index(np.argmax(info_grid), info_grid.shape)
best_x, best_y = XX[i_best, j_best], YY[i_best, j_best]
print(f"\nMejor posición: ({best_x/1000:+.2f}, {best_y/1000:+.2f}) km del cráter")
print(f"  info = {info_grid[i_best, j_best]:.4f}")
print(f"  L_max = {L_max_grid[i_best, j_best]:.0f} g/cm²")
print(f"  altitud del terreno = {det_z_grid[i_best, j_best]:.0f} m")

## 4. Heatmap del info-score sobre el DEM

DEM como fondo (hillshade), heatmap de información encima como overlay semitransparente.
El cráter va con estrella roja, el máximo de información con marker amarillo, la estación
default de INSIVUMEH con cuadrado cian (referencia para comparar).

In [ ]:
# DEM background ±15 km (más amplio que el grid para contexto)
ext = 15_000
xs_bg = np.linspace(-ext, ext, 500)
ys_bg = np.linspace(-ext, ext, 500)
XB, YB = np.meshgrid(xs_bg, ys_bg)
Z_bg = terrain.elevation_at(XB, YB)
ls = LightSource(azdeg=315, altdeg=45)
rgb_bg = ls.shade(Z_bg, cmap=plt.cm.terrain, vert_exag=2.0, blend_mode='overlay')

# Default station coords (relativas al cráter)
sites_df = pd.read_csv(REPO_ROOT / "data" / "detector_sites.csv")
_default = sites_df[(sites_df['name'] == VOLCAN_CFG['default_station']) &
                    (sites_df['target_volcan'] == TARGET_VOLCAN)].iloc[0]
_tx = Transformer.from_crs("EPSG:4326", f"EPSG:{terrain.utm_epsg}", always_xy=True)
_e, _n = _tx.transform(_default['lon'], _default['lat'])
default_x = _e - terrain.origin_utm[0]
default_y = _n - terrain.origin_utm[1]

fig, ax = plt.subplots(figsize=(11, 10))
ax.imshow(rgb_bg, extent=[-ext/1000, ext/1000]*2, origin='lower')

# Overlay del info-score (sólo en la región del grid)
heatmap = ax.pcolormesh(XX/1000, YY/1000, info_grid, cmap='magma',
                        alpha=0.65, shading='auto')
plt.colorbar(heatmap, ax=ax, label='Información = std(T) sobre píxeles del volcán',
             shrink=0.7)

# Markers
ax.scatter(0, 0, marker='*', s=420, color='red', edgecolor='black', lw=1.5, zorder=10,
           label=f"Cráter {TARGET_VOLCAN.title()}")
ax.scatter(best_x/1000, best_y/1000, marker='X', s=240, color='gold',
           edgecolor='black', lw=1.5, zorder=10,
           label=f'Óptimo ({info_grid[i_best, j_best]:.3f})')
ax.scatter(default_x/1000, default_y/1000, marker='s', s=160, color='cyan',
           edgecolor='black', lw=1.5, zorder=10,
           label=f"Default ({VOLCAN_CFG['default_station']})")

ax.set_xlabel('Easting from crater [km]')
ax.set_ylabel('Northing from crater [km]')
ax.set_title(f'Muogram information vs detector position ({TARGET_VOLCAN.title()})\n'
             f'Mayor = más contraste en T sobre el volcán = más señal tomográfica')
ax.set_aspect('equal')
ax.legend(loc='upper right')
ax.set_xlim(-ext/1000, ext/1000)
ax.set_ylim(-ext/1000, ext/1000)
plt.tight_layout()
fig.savefig(REPO_ROOT / 'docs/paper/figures' / f'fig_detector_optim_{TARGET_VOLCAN}.png',
            bbox_inches='tight', dpi=300)
plt.show()

## 5. Comparación: óptimo vs default

¿Cuánta mejora ofrece el óptimo respecto a la estación default? Y ¿es esa posición
geográficamente accesible? Ojo: el óptimo lo elige *únicamente* la métrica matemática,
sin saber nada de logística (carreteras, propiedad de la tierra, peligro volcánico).

In [ ]:
# Info-score at the default station position
default_z_ground = float(terrain.elevation_at(np.array(default_x), np.array(default_y)))
default_xyz = (default_x, default_y, default_z_ground + 2.0)
L_default = rock_opacity_grid(terrain, default_xyz, theta_rad_opt, phi_rad_opt,
                              max_length_m=8_000.0, n_steps=200)
T_default = transmission_map(L_default, theta_rad_opt)
info_default = muograma_information(T_default, L_default, mode='variance_volcano')
info_best = float(info_grid[i_best, j_best])

print(f"Default ({VOLCAN_CFG['default_station']}):")
print(f"  posición: ({default_x/1000:+.2f}, {default_y/1000:+.2f}) km")
print(f"  info: {info_default:.4f}")
print(f"  L max: {L_default.max():.0f} g/cm²")
print()
print(f"Óptimo del grid:")
print(f"  posición: ({best_x/1000:+.2f}, {best_y/1000:+.2f}) km")
print(f"  info: {info_best:.4f}")
print(f"  L max: {L_max_grid[i_best, j_best]:.0f} g/cm²")
print()
ratio = info_best / info_default if info_default > 0 else float('inf')
print(f"Mejora relativa: {ratio:.2f}× (óptimo / default)")
print()
print("Caveats:")
print("- El óptimo es la mejor posición SOLAMENTE según info=std(T). No considera:")
print("  - acceso vial / propiedad de la tierra")
print("  - actividad volcánica (¿esa posición es segura?)")
print("  - infraestructura existente (lo que motivó usar INSIVUMEH)")
print("- Si el óptimo cae cerca del default (mejora <20%), la estación INSIVUMEH es suficiente.")
print("- Si el óptimo es muy distinto, vale la pena explorar candidatos cercanos en el mapa.")

## Notas para extender

- **Grid más fino**: cambiá `GRID_N` o `HALF_EXTENT_KM`. El cómputo escala como N².
- **Otra métrica**: `muograma_information(..., mode='fraction_blocked')` u `'mean_deviation'`.
- **Múltiples detectores**: la información combinada de 2 detectores no es la suma — habría que computar una métrica conjunta tipo entropía mutua, no implementado acá.
- **Restricciones de accesibilidad**: enmascarar el `info_grid` con un mapa de áreas accesibles (carreteras, tierra de INSIVUMEH/INAB, etc.) antes de tomar el argmax.